# 12 — Node sequence, textual summary, and molecule fingerprint feature tables

Read-only summary notebook for the staged and promoted node feature-table waves:

- sequence-like nodes with source-backed sequences (`protein_sequence`, `transcript_sequence`);
- nodes with acceptable text descriptions (`*_textual_summary` feature tables);
- deterministic molecule structural fingerprints (`molecule_fingerprint`);
- explicit deferred gene genomic sequence status (`gene_sequence` / `gene_genomic_sequence` / `gene_genomic_interval`);
- validation status and the reviewer-approved / officially promoted feature-layer inclusion set.

This notebook is independent of ReMap. It documents model feature tables under `features/`; it does not create biological KG edges, evidence rows, or canonical outputs.


## Safety contract

Default behavior is safe and lightweight:

- `READ_ONLY = True`; the notebook only reads small local Markdown/JSON/CSV reports.
- No downloads, GCS writes, canonical feature promotion, biological `edges/`, or `evidence/` writes.
- No ReMap dependency; validation and promotion reports include explicit ReMap/edge/evidence negative checks.
- No heavy Parquet scans by default; staged/canonical Parquet paths are reported from existing manifests/validation reports.


In [ ]:
from __future__ import annotations

import csv
import json
from pathlib import Path
from typing import Any

import pandas as pd

try:
    from IPython.display import Markdown, display
except Exception:  # noqa: BLE001 - keep plain Python execution usable
    Markdown = None
    def display(obj):
        print(obj)

READ_ONLY = True
REPO_ROOT = Path.cwd()
DOCS = REPO_ROOT / 'docs'
CANONICAL_KG_ROOT = 'gs://jouvencekb/kg/v2'
FUSE_KG_ROOT = Path('/Users/jkobject/mnt/gcs/jouvencekb-kg/v2')
SEQUENCE_ROOT = 'gs://jouvencekb/kg/staging/node-sequence-features-20260622-t_f9ef6389'
TEXTUAL_ROOT = 'gs://jouvencekb/kg/staging/textual-summary-features-20260622-t_1b89d078'
MISSING_FEATURE_ROOT = 'gs://jouvencekb/kg/staging/node-missing-features-20260623-t_d6c55414'
VALIDATION_REPORT = DOCS / 'node_feature_tables_validation_report.md'
MISSING_VALIDATION_REPORT = DOCS / 'missing_node_feature_tables_validation_report.md'
MISSING_PROMOTION_MANIFEST = DOCS / 'missing_node_feature_tables_official_promotion_report.md'
PROTEIN_TEXT_PROMOTION_MANIFEST = DOCS / 'protein_textual_summary_official_promotion_report.md'

print('read_only:', READ_ONLY)
print('repo_root:', REPO_ROOT)
print('sequence_root:', SEQUENCE_ROOT)
print('textual_root:', TEXTUAL_ROOT)
print('missing_feature_root:', MISSING_FEATURE_ROOT)
print('validation_report:', VALIDATION_REPORT)
print('missing_validation_report:', MISSING_VALIDATION_REPORT)
print('missing_promotion_manifest:', MISSING_PROMOTION_MANIFEST)
print('protein_text_promotion_manifest:', PROTEIN_TEXT_PROMOTION_MANIFEST)


## Input reports and source documents

The notebook summarizes already-generated reports. Missing optional reports are surfaced as empty tables/errors rather than triggering downloads or rebuilds.


In [ ]:
def read_json(path: Path) -> dict[str, Any]:
    if not path.exists():
        return {'_missing': str(path)}
    return json.loads(path.read_text())


def read_csv_rows(path: Path) -> list[dict[str, str]]:
    if not path.exists():
        return []
    with path.open(newline='') as handle:
        return list(csv.DictReader(handle))

sequence_report = read_json(SEQUENCE_ROOT / 'reports/sequence_feature_report.json')
textual_report = read_json(TEXTUAL_ROOT / 'reports/textual_summary_features_summary.json')
validation = read_json(VALIDATION_REPORT)
missing_validation = read_json(MISSING_VALIDATION_REPORT)
missing_promotion = read_json(MISSING_PROMOTION_MANIFEST)
protein_text_promotion = read_json(PROTEIN_TEXT_PROMOTION_MANIFEST)
source_audit = read_csv_rows(TEXTUAL_ROOT / 'reports/textual_summary_source_audit.csv')

inputs = pd.DataFrame([
    {'kind': 'sequence source doc', 'path': 'docs/node_sequence_feature_sources.md', 'exists': (DOCS / 'node_sequence_feature_sources.md').exists()},
    {'kind': 'missing feature source doc', 'path': 'docs/node_feature_missing_gene_sequence_molecule_fingerprint_sources.md', 'exists': (DOCS / 'node_feature_missing_gene_sequence_molecule_fingerprint_sources.md').exists()},
    {'kind': 'textual source doc', 'path': 'docs/textual_summary_features.md', 'exists': (DOCS / 'textual_summary_features.md').exists()},
    {'kind': 'textual expansion plan', 'path': 'docs/textual_summary_features_expansion_plan.md', 'exists': (DOCS / 'textual_summary_features_expansion_plan.md').exists()},
    {'kind': 'sequence staging report', 'path': str(SEQUENCE_ROOT.relative_to(REPO_ROOT) / 'reports/sequence_feature_report.json'), 'exists': (SEQUENCE_ROOT / 'reports/sequence_feature_report.json').exists()},
    {'kind': 'textual staging report', 'path': str(TEXTUAL_ROOT.relative_to(REPO_ROOT) / 'reports/textual_summary_features_summary.json'), 'exists': (TEXTUAL_ROOT / 'reports/textual_summary_features_summary.json').exists()},
    {'kind': 'sequence/textual validation JSON', 'path': str(VALIDATION_REPORT.relative_to(REPO_ROOT)), 'exists': VALIDATION_REPORT.exists()},
    {'kind': 'missing-feature validation JSON', 'path': str(MISSING_VALIDATION_REPORT.relative_to(REPO_ROOT)), 'exists': MISSING_VALIDATION_REPORT.exists()},
    {'kind': 'missing-feature official promotion manifest', 'path': str(MISSING_PROMOTION_MANIFEST.relative_to(REPO_ROOT)), 'exists': MISSING_PROMOTION_MANIFEST.exists()},
    {'kind': 'protein text official promotion manifest', 'path': str(PROTEIN_TEXT_PROMOTION_MANIFEST.relative_to(REPO_ROOT)), 'exists': PROTEIN_TEXT_PROMOTION_MANIFEST.exists()},
    {'kind': 'textual source audit CSV', 'path': str(TEXTUAL_ROOT.relative_to(REPO_ROOT) / 'reports/textual_summary_source_audit.csv'), 'exists': (TEXTUAL_ROOT / 'reports/textual_summary_source_audit.csv').exists()},
])
display(inputs)


## Feature table schema contracts

Sequence, textual, and molecule fingerprint tables are model feature tables under `features/`, not graph assertions. The schemas below are copied/summarized from the audited source docs and validation/promotion JSON. Deferred gene genomic sequence contracts are shown to define semantics, not to imply that a table exists.


In [ ]:
sequence_columns = [
    'feature_key', 'feature_table', 'node_id', 'node_type', 'sequence_kind', 'sequence',
    'length', 'alphabet', 'source', 'source_dataset', 'source_record_id', 'source_release',
    'provenance', 'license', 'citation', 'created_at', 'checksum_sha256',
]
textual_columns = [
    'feature_key', 'feature_table', 'node_id', 'node_type', 'summary_kind', 'summary_text',
    'source', 'source_dataset', 'source_record_id', 'provenance', 'license', 'citation',
    'release', 'created_at',
]
gene_genomic_sequence_columns = [
    'feature_key', 'feature_table', 'node_id', 'node_type', 'sequence_kind', 'sequence',
    'length', 'alphabet', 'chromosome', 'start_1based', 'end_1based', 'strand',
    'reference_build', 'source', 'source_dataset', 'source_record_id', 'source_release',
    'provenance', 'license', 'citation', 'created_at', 'checksum_sha256',
]
molecule_fingerprint_columns = (missing_validation.get('molecule_fingerprint') or {}).get('schema_columns') or [
    'feature_key', 'feature_table', 'node_id', 'node_type', 'fingerprint_kind',
    'fingerprint_format', 'on_bits', 'n_bits', 'radius', 'use_chirality', 'use_bond_types',
    'input_smiles', 'canonical_smiles_rdkit', 'input_smiles_field', 'inchikey', 'source',
    'source_dataset', 'source_record_id', 'source_release', 'rdkit_version',
    'invalid_smiles_policy', 'salt_mixture_policy', 'component_count', 'provenance',
    'license', 'citation', 'created_at', 'fingerprint_sha256',
]
contracts = pd.DataFrame([
    {'family': 'sequence', 'tables': 'protein_sequence, transcript_sequence', 'columns': ', '.join(sequence_columns), 'dedupe_key': 'feature_table|node_id|source|source_dataset|source_record_id|sequence_kind', 'required_validation': 'endpoint anti-join clean; non-empty uppercase sequence; strict alphabet; max length; checksum matches'},
    {'family': 'gene_genomic_sequence (deferred)', 'tables': 'gene_sequence alias only if sequence_kind=genomic_locus; prefer gene_genomic_sequence after policy gate', 'columns': ', '.join(gene_genomic_sequence_columns), 'dedupe_key': 'feature_table|node_id|source|source_dataset|source_record_id|reference_build|sequence_kind', 'required_validation': 'coordinate/source/reference-build/mapping/strand/length policy first; no transcript-derived shortcut; no placeholder parquet'},
    {'family': 'textual_summary', 'tables': '*_textual_summary', 'columns': ', '.join(textual_columns), 'dedupe_key': 'feature_table|node_id|source|source_dataset|source_record_id|summary_kind', 'required_validation': 'endpoint anti-join clean; node_type contract; non-empty summary_text; max text chars'},
    {'family': 'molecule_fingerprint', 'tables': 'molecule_fingerprint', 'columns': ', '.join(molecule_fingerprint_columns), 'dedupe_key': 'feature_table|node_id|source|source_dataset|source_record_id|fingerprint_kind|radius|n_bits|use_chirality', 'required_validation': 'endpoint anti-join clean; deterministic RDKit recomputation; non-empty sorted on_bits; checksum matches; source SMILES parity'},
])
display(contracts)


## Sequence source matrix

Current accepted/promoted sequence sources are Ensembl release 114 for conservative protein/transcript feature tables. Gene genomic sequence is explicitly deferred: a bare `gene_sequence` must not mean transcript-derived sequence or a placeholder. It should only be built after a reviewed coordinate/reference-build/mapping/strand/length policy, preferably via a `gene_genomic_interval` precursor and then an explicit `gene_genomic_sequence` table.


In [ ]:
sequence_matrix = pd.DataFrame([
    {
        'node_type': 'protein',
        'feature_table': 'protein_sequence',
        'source': 'Ensembl human protein FASTA (pep.all)',
        'release_or_version': 'Ensembl release 114 / GRCh38 / FASTA last-modified 2025-02-02',
        'license_terms': 'EMBL-EBI open data / attribution',
        'coverage': '112,051 / 233,995 nodes (47.89%)',
        'historical_local_staged_path': 'legacy .omoc/staging/node-sequence-features-20260622-t_f9ef6389/features/protein_sequence.parquet',
        'remote_staged_path': 'gs://jouvencekb/kg/staging/node-sequence-features-20260622-t_f9ef6389/features/protein_sequence.parquet',
        'official_path': 'gs://jouvencekb/kg/v2/features/protein_sequence.parquet',
        'decision': 'staged, validated, reviewer-approved, and officially promoted',
    },
    {
        'node_type': 'transcript',
        'feature_table': 'transcript_sequence',
        'source': 'Ensembl human cDNA FASTA (cdna.all)',
        'release_or_version': 'Ensembl release 114 / GRCh38 / FASTA last-modified 2025-02-02',
        'license_terms': 'EMBL-EBI open data / attribution',
        'coverage': '187,268 / 507,365 nodes (36.91%)',
        'historical_local_staged_path': 'legacy .omoc/staging/node-sequence-features-20260622-t_f9ef6389/features/transcript_sequence.parquet',
        'remote_staged_path': 'gs://jouvencekb/kg/staging/node-sequence-features-20260622-t_f9ef6389/features/transcript_sequence.parquet',
        'official_path': 'gs://jouvencekb/kg/v2/features/transcript_sequence.parquet',
        'decision': 'staged, validated, reviewer-approved, and officially promoted',
    },
    {
        'node_type': 'gene',
        'feature_table': 'gene_sequence / gene_genomic_sequence',
        'source': 'not built; would require Ensembl/GENCODE GTF/GFF3 coordinates + matching GRCh38 FASTA',
        'release_or_version': 'deferred; must pin annotation release, reference FASTA build/accession/checksum, coordinate convention, strand policy, and max-length policy',
        'license_terms': 'source-dependent; must preserve reference/source attribution',
        'coverage': '0 rows; no raw sequence payload staged',
        'local_staged_path': None,
        'remote_staged_path': None,
        'official_path': None,
        'decision': 'deferred; no placeholder parquet; do not synthesize from transcript_sequence; prefer gene_genomic_interval precursor before bounded raw genomic sequence',
    },
    {'node_type': 'protein', 'feature_table': 'future protein_uniprot_sequence or replacement', 'source': 'UniProtKB FASTA/REST', 'release_or_version': 'not used in this wave', 'license_terms': 'CC BY 4.0', 'coverage': 'not built', 'local_staged_path': None, 'remote_staged_path': None, 'official_path': None, 'decision': 'defer until source-policy gate decides Ensembl vs UniProt canonical sequence source'},
    {'node_type': 'enhancer/mutation/miRNA/lncRNA', 'feature_table': 'deferred sequence tables', 'source': 'reference genome / miRBase / GENCODE / source-specific inputs', 'release_or_version': 'not built', 'license_terms': 'source-dependent', 'coverage': 'not built', 'local_staged_path': None, 'remote_staged_path': None, 'official_path': None, 'decision': 'defer; requires coordinate, node-ID, and license policy before sequence feature creation'},
])
display(sequence_matrix)


## Textual summary source matrix

Textual summaries use only sources with acceptable redistribution/attribution terms. GeneCards, DrugBank text scraping, Orphanet, Reactome web-page scraping, paper abstracts, and dataset prose remain rejected/deferred unless a later terms review accepts them.


In [ ]:
textual_matrix = pd.DataFrame([
    {'node_type': 'gene', 'feature_table': 'gene_textual_summary', 'source': 'OpenTargets target descriptions with upstream attribution', 'release_or_version': '2026-06-22-local-cache-expanded', 'license_terms': 'CC BY 4.0 / upstream source attribution', 'coverage': '212,029 / 267,830 (79.17%)', 'historical_local_staged_path': 'legacy .omoc/staging/textual-summary-features-20260622-t_1b89d078/features/gene_textual_summary.parquet', 'remote_staged_path': 'gs://jouvencekb/kg/staging/textual-summary-features-20260622-t_1b89d078/features/gene_textual_summary.parquet', 'decision': 'accepted staged; GeneCards rejected'},
    {'node_type': 'protein', 'feature_table': 'protein_textual_summary', 'source': 'UniProtKB accepted comment blocks', 'release_or_version': '2026-06-23-full-uniprot-local-cache', 'license_terms': 'CC BY 4.0', 'coverage': '162,163 / 233,995 (69.301908%)', 'historical_local_staged_path': 'legacy .omoc/tmp/protein_textual_summary_promotion_t_5fbd4051', 'remote_staged_path': 'gs://jouvencekb/kg/v2/features/protein_textual_summary.parquet', 'decision': 'officially promoted full UniProt feature; earlier 228-row pilot is historical/superseded'},
    {'node_type': 'disease', 'feature_table': 'disease_textual_summary', 'source': 'OpenTargets disease descriptions / EFO-MONDO attribution', 'release_or_version': '2026-06-22-local-cache-expanded', 'license_terms': 'CC BY 4.0 / ontology attribution', 'coverage': '26,395 / 41,859 (63.06%)', 'historical_local_staged_path': 'legacy .omoc/staging/textual-summary-features-20260622-t_1b89d078/features/disease_textual_summary.parquet', 'remote_staged_path': 'gs://jouvencekb/kg/staging/textual-summary-features-20260622-t_1b89d078/features/disease_textual_summary.parquet', 'decision': 'accepted staged; Orphanet deferred'},
    {'node_type': 'tissue', 'feature_table': 'tissue_textual_summary', 'source': 'UBERON def fields', 'release_or_version': 'UBERON 2026-04-01 observed locally', 'license_terms': 'CC BY 4.0 / ontology attribution', 'coverage': '11,942 / 16,061 (74.35%)', 'historical_local_staged_path': 'legacy .omoc/staging/textual-summary-features-20260622-t_1b89d078/features/tissue_textual_summary.parquet', 'remote_staged_path': 'gs://jouvencekb/kg/staging/textual-summary-features-20260622-t_1b89d078/features/tissue_textual_summary.parquet', 'decision': 'accepted staged'},
    {'node_type': 'molecule', 'feature_table': 'molecule_textual_summary', 'source': 'ChEMBL-derived OpenTargets molecule metadata', 'release_or_version': '2026-06-22-local-cache-expanded', 'license_terms': 'CC BY-SA 3.0 / OpenTargets attribution', 'coverage': '22,230 / 31,007 (71.69%)', 'historical_local_staged_path': 'legacy .omoc/staging/textual-summary-features-20260622-t_1b89d078/features/molecule_textual_summary.parquet', 'remote_staged_path': 'gs://jouvencekb/kg/staging/textual-summary-features-20260622-t_1b89d078/features/molecule_textual_summary.parquet', 'decision': 'accepted staged; DrugBank descriptions rejected/deferred'},
    {'node_type': 'pathway', 'feature_table': 'pathway_textual_summary', 'source': 'GO def fields', 'release_or_version': '2026-06-22-local-cache-expanded', 'license_terms': 'CC BY 4.0', 'coverage': '37,492 / 48,575 (77.18%)', 'historical_local_staged_path': 'legacy .omoc/staging/textual-summary-features-20260622-t_1b89d078/features/pathway_textual_summary.parquet', 'remote_staged_path': 'gs://jouvencekb/kg/staging/textual-summary-features-20260622-t_1b89d078/features/pathway_textual_summary.parquet', 'decision': 'accepted staged; Reactome web scraping rejected, accepted dump/API deferred until local payload exists'},
    {'node_type': 'cell_type', 'feature_table': 'cell_type_textual_summary', 'source': 'Cell Ontology def fields', 'release_or_version': 'CL releases/2026-06-08 observed locally', 'license_terms': 'CC BY 4.0 / ontology attribution', 'coverage': '3,135 / 3,513 (89.24%)', 'historical_local_staged_path': 'legacy .omoc/staging/textual-summary-features-20260622-t_1b89d078/features/cell_type_textual_summary.parquet', 'remote_staged_path': 'gs://jouvencekb/kg/staging/textual-summary-features-20260622-t_1b89d078/features/cell_type_textual_summary.parquet', 'decision': 'accepted staged'},
    {'node_type': 'phenotype', 'feature_table': 'phenotype_textual_summary', 'source': 'HPO def fields', 'release_or_version': 'HPO 2026-06-06 observed locally', 'license_terms': 'HPO license / attribution required', 'coverage': '13,810 / 16,449 (83.96%)', 'historical_local_staged_path': 'legacy .omoc/staging/textual-summary-features-20260622-t_1b89d078/features/phenotype_textual_summary.parquet', 'remote_staged_path': 'gs://jouvencekb/kg/staging/textual-summary-features-20260622-t_1b89d078/features/phenotype_textual_summary.parquet', 'decision': 'accepted staged for HP IDs only'},
    {'node_type': 'cell_line', 'feature_table': 'cell_line_textual_summary', 'source': 'Cellosaurus OBO comments via DepMap ACH xrefs', 'release_or_version': '55.0; date=03:31:2026 12:00', 'license_terms': 'CC BY 4.0 with attribution/link/change notice', 'coverage': '1,140 / 1,183 (96.37%)', 'historical_local_staged_path': 'legacy .omoc/staging/textual-summary-features-20260622-t_1b89d078/features/cell_line_textual_summary.parquet', 'remote_staged_path': 'gs://jouvencekb/kg/staging/textual-summary-features-20260622-t_1b89d078/features/cell_line_textual_summary.parquet', 'decision': 'accepted staged after license metadata fix'},
    {'node_type': 'organism/dataset/paper', 'feature_table': 'optional/deferred textual tables', 'source': 'NCBI Taxonomy / project-specific metadata / PubMed-Crossref-OpenAlex', 'release_or_version': 'not built', 'license_terms': 'source-dependent', 'coverage': 'not built', 'local_staged_path': None, 'remote_staged_path': None, 'decision': 'defer until node tables and license/text-field policy are explicit'},
])
display(textual_matrix)


## Molecule fingerprint source matrix

`molecule_fingerprint` is a molecule structural feature, not a textual summary and not a biological edge. It is computed from `nodes/molecule.parquet.smiles` with deterministic RDKit Morgan parameters, staged/validated in the missing-feature wave, then reviewer-approved and officially promoted as the only object from that wave. Missing/invalid structures do not receive fabricated all-zero rows.


In [ ]:
mf = missing_validation.get('molecule_fingerprint') or {}
fixed = mf.get('fixed_parameter_values') or {}
source_policy = mf.get('source_policy_rows') or []
molecule_fingerprint_matrix = pd.DataFrame([
    {
        'node_type': 'molecule',
        'feature_table': 'molecule_fingerprint',
        'source': 'nodes/molecule.parquet.smiles from ChEMBL/OpenTargets molecule metadata',
        'release_or_version': f"KG molecule source release from feature rows; RDKit {mf.get('fixed_parameter_values', {}).get('rdkit_version', ['2026.03.3'])[0] if fixed else '2026.03.3'}",
        'license_terms': '; '.join(f"{row.get('source')}: {row.get('license')}" for row in source_policy) or 'ChEMBL/OpenTargets attribution captured in feature rows',
        'coverage': f"{mf.get('rows', 18614):,} / {mf.get('endpoint_nodes', 31007):,} molecule nodes ({100 * mf.get('coverage_fraction', 0.6003160576643983):.2f}%)",
        'historical_local_staged_path': 'legacy .omoc/staging/node-missing-features-20260623-t_d6c55414/features/molecule_fingerprint.parquet',
        'remote_staged_path': 'gs://jouvencekb/kg/staging/node-missing-features-20260623-t_d6c55414/features/molecule_fingerprint.parquet',
        'official_path': 'gs://jouvencekb/kg/v2/features/molecule_fingerprint.parquet',
        'decision': 'staged, validated, reviewer-approved, and officially promoted; no gene_sequence or gene_genomic_interval promoted in this wave',
    }
])
display(molecule_fingerprint_matrix)

molecule_fingerprint_parameters = pd.DataFrame([
    {'parameter': key, 'value': ', '.join(map(str, value)) if isinstance(value, list) else value}
    for key, value in sorted(fixed.items())
])
display(molecule_fingerprint_parameters)

molecule_fingerprint_caveats = pd.DataFrame([
    {'caveat': 'SMILES source', 'status': 'input_smiles_field verified as nodes/molecule.parquet.smiles; source SMILES mismatch rows = 0'},
    {'caveat': 'missing structures', 'status': f"{mf.get('missing_source_smiles_count', 12393):,} molecule nodes lacked non-empty SMILES and were skipped, not filled with placeholders"},
    {'caveat': 'invalid SMILES', 'status': f"invalid SMILES report rows = {mf.get('invalid_smiles_report_rows', 0)}; invalid policy = skip_with_report"},
    {'caveat': 'salts/mixtures', 'status': f"{mf.get('multi_component_rows', 1918):,} multi-component rows fingerprinted as input SMILES; component_count recorded"},
    {'caveat': 'determinism', 'status': f"full RDKit recomputation rows = {mf.get('full_recompute_rows', 18614):,}; mismatches = {mf.get('full_recompute_mismatch_count', 0)}; checksum mismatches = {mf.get('checksum_mismatch_rows', 0)}"},
])
display(molecule_fingerprint_caveats)


## Validation summary

The original sequence/textual validation gate passed on staged artifacts only. The later missing-feature validation passed `molecule_fingerprint` and explicitly passed the `gene_sequence` deferral decision: no raw `gene_sequence` payload, no `gene_genomic_interval` placeholder, and no canonical writes from validation. Official promotion was handled by separate reviewer-approved promotion gates.


In [ ]:
def percent(value: float | None) -> str | None:
    if value is None:
        return None
    return f'{100 * value:.2f}%'

sequence_rows = []
for table, data in (validation.get('sequence_tables') or {}).items():
    v = data.get('validation', {})
    sequence_rows.append({
        'table': table,
        'status': data.get('status'),
        'rows': v.get('rows'),
        'unique_nodes': v.get('unique_nodes'),
        'endpoint_nodes': v.get('endpoint_nodes'),
        'coverage': percent(v.get('coverage_fraction')),
        'max_length': v.get('max_length'),
        'sources': ', '.join(data.get('sources', [])),
        'nodes_not_in_endpoint': v.get('nodes_not_in_endpoint'),
        'invalid_alphabet_rows': v.get('invalid_alphabet_rows'),
        'over_max_length_rows': v.get('over_max_length_rows'),
    })
textual_rows = []
for table, data in (validation.get('textual_tables') or {}).items():
    v = data.get('validation', {})
    textual_rows.append({
        'table': table,
        'status': data.get('status'),
        'rows': v.get('rows'),
        'unique_nodes': v.get('unique_nodes'),
        'endpoint_nodes': v.get('endpoint_nodes'),
        'coverage': percent(v.get('coverage_fraction')),
        'max_chars': data.get('max_text_length_observed'),
        'sources': ', '.join(data.get('sources', [])),
        'nodes_not_in_endpoint': v.get('nodes_not_in_endpoint'),
        'missing_summary_text_rows': v.get('missing_summary_text_rows'),
        'over_max_text_rows': v.get('over_max_text_rows'),
    })

print('sequence_textual_overall_status:', validation.get('overall_status'))
print('missing_feature_verdict:', missing_validation.get('verdict'))
print('canonical_promotion_authorized_by_validation:', False)
print('canonical_promotion_performed_by_validation:', False)
print('remap_scan:', validation.get('remap_scan'))
print('negative_prefix_checks:', validation.get('local_prefix_checks'))
print('gcs_negative_checks:', {k: {field: v.get(field) for field in ['feature_object_count', 'edges_absent_remote', 'evidence_absent_remote', 'status']} for k, v in (validation.get('gcs_checks') or {}).items()})

print('Sequence tables')
display(pd.DataFrame(sequence_rows))
print('Textual summary tables')
display(pd.DataFrame(textual_rows).sort_values('table'))

mf = missing_validation.get('molecule_fingerprint') or {}
gs = missing_validation.get('gene_sequence') or {}
missing_feature_rows = pd.DataFrame([
    {
        'table': 'molecule_fingerprint',
        'validation_status': missing_validation.get('verdict'),
        'rows': mf.get('rows'),
        'unique_nodes': mf.get('unique_nodes'),
        'endpoint_nodes': mf.get('endpoint_nodes'),
        'coverage': percent(mf.get('coverage_fraction')),
        'nodes_not_in_endpoint': mf.get('nodes_not_in_endpoint_count'),
        'checksum_mismatch_rows': mf.get('checksum_mismatch_rows'),
        'full_recompute_mismatch_count': mf.get('full_recompute_mismatch_count'),
        'promotion_status': 'reviewer-approved and officially promoted by t_607adb48',
    },
    {
        'table': 'gene_sequence / gene_genomic_interval',
        'validation_status': 'PASS for deferral decision',
        'rows': 0,
        'unique_nodes': 0,
        'endpoint_nodes': None,
        'coverage': '0.00% / not built',
        'nodes_not_in_endpoint': None,
        'checksum_mismatch_rows': 'not applicable: no sequence payload',
        'full_recompute_mismatch_count': 'not applicable',
        'promotion_status': 'deferred; explicitly not promoted',
    },
])
display(missing_feature_rows)

print('missing_feature_recommendation:', missing_validation.get('recommendation'))


## Official feature-layer inclusion status

The original validator report records a staged/schema-clean set and is not itself an official inclusion gate. The current official feature layer is the product of reviewer-approved promotion gates:

- first 10 sequence/textual tables from `docs/node_feature_tables_official_promotion_report.md`;
- `molecule_fingerprint` from `docs/missing_node_feature_tables_official_promotion_report.md`;
- `protein_textual_summary` from `docs/protein_textual_summary_official_promotion_report.md`.

`gene_sequence` / `gene_genomic_interval` remains deferred and explicitly not promoted. This notebook is read-only and does not perform or approve canonical promotion.


In [ ]:
validator_recommended = validation.get('recommended_feature_inclusion_set', [])
validator_recommended_paths = [f'features/{name}.parquet' for name in validator_recommended]
validator_recommended_df = pd.DataFrame({
    'validator_staged_schema_clean_table': validator_recommended_paths,
    'validator_scope': 'schema-clean staged candidate; not the official inclusion gate',
})
display(validator_recommended_df)

reviewer_approved_official_feature_tables = [
    'features/cell_line_textual_summary.parquet',
    'features/cell_type_textual_summary.parquet',
    'features/disease_textual_summary.parquet',
    'features/gene_textual_summary.parquet',
    'features/molecule_fingerprint.parquet',
    'features/molecule_textual_summary.parquet',
    'features/pathway_textual_summary.parquet',
    'features/phenotype_textual_summary.parquet',
    'features/protein_sequence.parquet',
    'features/protein_textual_summary.parquet',
    'features/tissue_textual_summary.parquet',
    'features/transcript_sequence.parquet',
]
official_inclusion_df = pd.DataFrame({
    'official_feature_table': reviewer_approved_official_feature_tables,
    'decision': 'reviewer-approved and officially promoted to gs://jouvencekb/kg/v2/features/',
})
display(official_inclusion_df)

deferred_feature_tables = pd.DataFrame([
    {
        'deferred_feature_table': 'features/gene_sequence.parquet',
        'current_status': 'not staged; not validated as a payload; not official inclusion',
        'reason': 'Bare gene_sequence is biologically ambiguous; no reviewed coordinate/reference-build/mapping/strand/length policy exists; must not be transcript-derived or placeholder.',
        'follow_up': 'build/review gene_genomic_interval precursor, then bounded gene_genomic_sequence only after policy acceptance',
    },
    {
        'deferred_feature_table': 'features/gene_genomic_interval.parquet',
        'current_status': 'not staged; not promoted',
        'reason': 'No reviewed GTF/GFF coordinate source supplied for current KG gene IDs.',
        'follow_up': 'coordinate/source mapping audit before raw gene genomic sequence extraction',
    },
])
display(deferred_feature_tables)

feature_status = pd.DataFrame([
    {'feature': 'molecule_fingerprint', 'staged': True, 'validated': True, 'reviewer_approved': True, 'officially_promoted': True, 'deferred': False, 'notes': '18,614 rows; RDKit Morgan radius 2 / 2048 bits / chirality and bond types; source SMILES nodes/molecule.parquet.smiles'},
    {'feature': 'gene_sequence', 'staged': False, 'validated': 'deferral only', 'reviewer_approved': False, 'officially_promoted': False, 'deferred': True, 'notes': 'No raw sequence payload; requires reviewed coordinates, GRCh38/reference FASTA, mapping, strand, checksum, alphabet, max-length policy'},
])
display(feature_status)

print(
    'Official status: 12 reviewer-approved feature tables are promoted under kg/v2/features. '
    'molecule_fingerprint is included; gene_sequence/gene_genomic_interval remain deferred. '
    'This read-only notebook performs no canonical writes or promotion approval.'
)


## Explicit exclusions and non-dependencies

- No ReMap dependency: validation/promotion scans reported no ReMap-named feature outputs, and ReMap is unrelated to molecule fingerprints or gene genomic sequence policy.
- No biological `edges/` or `evidence/` writes: local and GCS negative checks passed for sequence/textual staging and the missing-feature promotion kept edge/evidence listing counts unchanged.
- No canonical feature-layer writes in this notebook or the validation gates. Promotion is documented from existing reviewer-approved reports only.
- `gene_sequence` is not a synonym for transcript/cDNA sequence in this contract. Raw gene genomic sequence remains deferred until coordinate/source/reference-build/strand/alphabet/checksum/maximum-length policy is reviewed.
- Rejected/deferred text sources remain outside the feature wave: GeneCards, DrugBank textual scraping, Orphanet, Reactome web-page scraping, and paper/dataset prose without an accepted text-field/license policy.


In [ ]:
policy = pd.DataFrame(source_audit)
if not policy.empty:
    display(policy[['source', 'decision', 'license', 'reason']].sort_values(['decision', 'source']))
else:
    print('No textual source audit CSV found.')
